# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZiadYakout/FlyRank-Ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


For the Refresh / Content Opportunity Scoring lane, the final decision-time unit of analysis is **one content item for one client at a March 2026 snapshot**.

The underlying warehouse table is `fact_content_daily_performance`, whose daily grain is one row per **report_date × client_id × content_id**.

For this contract, I will use the March 2026 partition as a mid-panel development window. I will aggregate the daily observations to one row per **client_id × content_id** for the feature frame.

The March window is used for development and is not treated as the final test window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

# Create a DuckDB connection
con = duckdb.connect()

# Load the HTTP filesystem extension for the hosted warehouse
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

print("DuckDB connection ready.")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
import duckdb


### Features

The initial feature frame will contain at most five decision-time features:

- `impressions` — search exposure available before the decision.
- `clicks` — search clicks observed before the decision.
- `sessions` — sessions observed before the decision, when GA4 data is available.
- `gsc_avg_position` — observed search position available before the decision.
- `days_since_last_update` — content freshness information available at the decision moment.

### Label

The eventual label will be an **observed future content outcome** measured after the feature window. The exact outcome and threshold will be defined before model training.

The March 2026 feature window itself will not be used as its own future label window.

### Context

The following fields are useful for understanding or grouping the data but are not model features in the initial five-feature frame:

- `client_id` — identifies the client and can be used for grouping or validation.
- `content_id` — identifies the content item and is used for joins/grouping.
- `report_date` — identifies the daily observation date.
- Data availability flags — used to determine whether the corresponding measurements are usable.

### Excluded

I deliberately exclude `trend_direction` and `trend_pct` from the honest feature set because the FlyRank data guidance states that the decline label is derived from these signals. Using them as features could leak information about the outcome into the model.

I also exclude pseudonymous IDs such as `client_id` and `content_id` from model features because they identify entities rather than represent meaningful predictive signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
schema = con.execute("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

schema

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


The following checks verify the contract against the March 2026 warehouse partition.

First, I check whether the documented daily grain has duplicate rows. Next, I check the number of rows and date range. I then check data availability using explicit boolean checks and inspect missingness in the fields used for the feature frame.

These checks are intended to verify the contract rather than assume that the documentation alone is sufficient.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Path to the March 2026 warehouse partition
march_path = """
hf://datasets/FlyRank/internship-warehouse/
fact_content_daily_performance/month=2026-03/*.parquet
"""

# ---------------------------------------------------------
# 1. GRAIN CHECK
# One row should represent:
# report_date × client_id × content_id
# ---------------------------------------------------------

grain_check = con.execute("""
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS n
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    report_date,
    client_id,
    content_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate grain combinations:", len(grain_check))
grain_check

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This data has several important limitations.

**Uneven history:** Different clients have different amounts of historical data. Therefore, a single global calendar window does not necessarily represent the same amount of history for every client.

**GSC-only early rows:** Some clients have Google Search Console data before their GA4 data becomes available. GA4 availability must therefore be checked explicitly rather than treating zero-filled GA4 values as genuine zero engagement.

**Window overlap:** The warehouse query table contains fixed 90-day context that can overlap with later outcome windows. Features must therefore be aligned carefully with the intended label window so that future information is not accidentally included.

**Final-month limitation:** The final month should be treated as a sealed test window. March 2026 is being used here as a mid-panel development window.

**Decision-support limitation:** This data can support observed and directional analysis, but it cannot by itself prove that changing a page will cause a particular outcome. The eventual model should therefore be treated as decision-support rather than causal proof.

**Availability limitation:** A row with missing or unavailable measurement should not automatically be interpreted as a true zero. Availability flags must be checked before using the corresponding measurements.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.